# 01 · Build — Medical-Information QA Assistant

**Vertical:** Healthcare · **Use case:** medical-information-qa · **ADLC stage:** Build

A medical-information team needs a **cited, safety-gated answer** over synthetic prescribing information:
answer on-label questions from sources, refuse off-label or personal-advice asks, escalate suspected
adverse events, and avoid repeating PII.

![Architecture](images/01_build.png)

> Self-directed module · ~10 min. By the end you can:
> - build a PI-grounded medical-information QA agent with `create_agent`
> - wire four local tools for inquiry lookup, drug metadata, PI search, and policy lookup
> - enforce a typed safety answer with `response_format=` instead of parsing free text
> - render the compiled graph and open traces / prompt in LangSmith
>
> 🧭 **First lab in the track — runs standalone.** All fixtures are **synthetic / fictional**: no real
> drugs, labels, patients, PII, secrets, medical advice, or pharmacovigilance action.

| Tool | Does |
|---|---|
| `get_inquiry` | retrieve a synthetic inquiry record by ID |
| `get_drug_profile` | retrieve fictional brand/generic, indication, contraindication tags, AE terms |
| `search_prescribing_info` | search synthetic PI + policy Markdown and return cited snippets |
| `check_safety_policy` | retrieve refusal, adverse-event, PII, and response policy snippets |

## 0. Setup

> 💡 `override=True` lets this lab's `.env` win over globally-set credentials. The printed
> **workspace** is where traces and Prompt Hub links route.

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)  # FIRST — before any model client is constructed
if "gateway.smith.langchain.com" in os.environ.get("OPENAI_BASE_URL", ""):
    os.environ.pop("OPENAI_BASE_URL", None)  # use the provider key directly, not an injected gateway

_root = Path.cwd()
while _root != _root.parent and not (_root / "utils").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))  # import utils/ and medical_information_qa_graph.py from the content root
DATA = _root / "data"

print("workspace:", os.getenv("LANGSMITH_WORKSPACE_ID"), "| tracing:", os.getenv("LANGSMITH_TRACING"))

## 1. Inspect the synthetic PI fixtures

The agent is allowed to ground itself only in committed fixtures: inquiry records, compact drug profiles,
fictional prescribing-information labels, and medical-information policies.

In [ ]:
inquiries = json.loads((DATA / "inquiries.json").read_text())
drugs = json.loads((DATA / "drug_profiles.json").read_text())
kb_docs = sorted((DATA / "kb").glob("*.md"))

print("inquiries:", len([k for k in inquiries if not k.startswith("_")]))
print("drugs    :", len([k for k in drugs if not k.startswith("_")]))
print("kb docs  :", [p.name for p in kb_docs])

In [ ]:
inquiry = inquiries["MIQ-001"]
print(inquiry["query"])
print("drug:", inquiry["drug_id"], "| disposition:", inquiry["expected_disposition"])
print("required citations:")
for item in inquiry["required_citations"]:
    print(" -", item)

## 2. Inspect the grounding tools

These four tools are pure-Python lookups/searches over the same fixtures — no clinical system, SQL,
network, or real pharmacovigilance action. Try each one before giving it to the agent.

In [ ]:
from medical_information_qa_graph import (
    check_safety_policy,
    get_drug_profile,
    get_inquiry,
    medical_information_qa_tools,
    search_prescribing_info,
)

tools = medical_information_qa_tools()
print("tools:", [t.name for t in tools])

In [ ]:
print(get_inquiry.invoke({"inquiry_id": "MIQ-001"})[:900])

In [ ]:
print(get_drug_profile.invoke({"query": "Alvexora"})[:800])

In [ ]:
print(search_prescribing_info.invoke({"query": "Alvexora starting dose adult chronic glimmerase deficiency"})[:1100])

In [ ]:
print(check_safety_policy.invoke({"topic": "off-label personal medical advice refusal"})[:1100])

## 3. Load the medical-information prompt

The prompt sets role, grounding rules, refusal taxonomy, adverse-event escalation, and PII handling. The
answer shape stays in code (§4), not in prompt formatting instructions.

In [ ]:
from medical_information_qa_graph import PROMPT_NAME, PROMPT_PATH
from utils.prompts import get_prompt, prompt_url

system_prompt = get_prompt(PROMPT_NAME, fallback=PROMPT_PATH.read_text().strip())
print(system_prompt)
print("\n📝 prompt:", prompt_url(PROMPT_NAME))

## 4. Build the agent

The answer's shape is a **Pydantic schema**, not a JSON/YAML template in the prompt. Passed as
`response_format=`, LangChain enforces the fields + allowed dispositions and returns a parsed object in
`result["structured_response"]`. **The schema is code.**

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class MedicalInformationAnswer(BaseModel):
    disposition: Literal[
        "answer", "answer_with_limitations", "refuse_off_label", "refuse_personal_advice", "escalate_adverse_event"
    ] = Field(description="The final response disposition after applying grounding and safety policy.")
    answer: str = Field(description="Concise cited answer, refusal, or escalation summary with no raw PII.")
    safety_flags: list[Literal["off_label", "personal_medical_advice", "adverse_event", "pii_redacted", "unsupported_claim"]] = Field(
        description="Safety flags detected in the request or needed for the answer."
    )
    escalate_adverse_event: bool = Field(description="True when the inquiry should route to medical review as a suspected adverse event.")
    pii_redacted: bool = Field(description="True when direct identifiers were present and must not be repeated.")
    redaction_notes: list[str] = Field(description="Types of identifiers redacted, never the raw identifier values.")
    citations: list[str] = Field(description="Fixture IDs / KB filenames used: inquiry IDs, drug IDs, structured JSON, kb *.md#section.")

One clean keyword-arg call: model + tools + prompt + the response schema in, a compiled ReAct agent out.

In [ ]:
from langchain.agents import create_agent

from utils.models import get_model

model = get_model()  # default model comes from utils/models.py; swap it once there

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    response_format=MedicalInformationAnswer,  # ← enforce the typed safety answer; no output template in the prompt
)

**The agent's shape** — `create_agent` returns a compiled LangGraph graph; render it inline.

> 💡 Adding `response_format` gives the `model` node a structured-output validation path before ending.
> Without it, the graph is just the plain `model ↔ tools → __end__` ReAct loop.

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## 5. Run one on-label PI answer

`ask()` invokes the agent and prints a clickable **LangSmith trace** link when tracing is enabled. Because
the agent has a `response_format`, `ask()` returns the validated `MedicalInformationAnswer` object.

In [ ]:
from utils.trace import ask

answer = ask(agent, "MIQ-001: What is Alvexora indicated for and what is the starting dose?")

In [ ]:
print("disposition :", answer.disposition)
print("flags       :", answer.safety_flags)
print("AE escalate :", answer.escalate_adverse_event)
print("PII redacted:", answer.pii_redacted)
print("\nanswer:\n", answer.answer)
print("\ncitations:")
for citation in answer.citations:
    print("  •", citation)

## 6. Check a safety-gated adverse-event case

A suspected adverse event is **not routine Q&A**. The agent should route it for medical review, preserve
non-identifying safety context, and avoid repeating placeholder identifiers.

In [ ]:
ae_answer = ask(
    agent,
    "MIQ-009: [FAKE_NAME: Patient Alpha] had facial swelling and hives after Elmurestat and went to urgent care. Their fake phone is 000-000-0000.",
)

In [ ]:
print("disposition :", ae_answer.disposition)
print("flags       :", ae_answer.safety_flags)
print("AE escalate :", ae_answer.escalate_adverse_event)
print("PII redacted:", ae_answer.pii_redacted)
print("redactions  :", ae_answer.redaction_notes)
print("\nanswer:\n", ae_answer.answer)

## 7. Recap

| Topic | API |
|---|---|
| PI-grounded ReAct agent | `create_agent(model=..., tools=..., system_prompt=...)` |
| Structured safety answer | `response_format=MedicalInformationAnswer` → `result["structured_response"]` |
| Local grounding tools | `get_inquiry`, `get_drug_profile`, `search_prescribing_info`, `check_safety_policy` |
| Trace + prompt links | `utils.trace.ask()` · `prompt_url(PROMPT_NAME)` |

**Next:** `02_evaluate.ipynb` — measure groundedness, refusal correctness, AE escalation, and PII safety.